In [ ]:
# -*- coding: utf-8 -*-
"""
TCNN SUAVE — Thermal Compensation Neural Network para EMI/SHM

Objetivo:
    curva medida Z(f, T, dano)  --->  curva equivalente em REF_TEMP,
    preservando a assinatura de dano.

Ideia principal:
    - Entrada da rede: somente a curva.
    - Temperatura NÃO entra como entrada.
    - Falha NÃO entra como entrada.
    - Temperatura e falha entram apenas como alvos auxiliares de treino.
    - A correção é residual: y_comp = x + alpha * delta.
    - O delta é forçado a ser suave para não apagar picos/vales locais do dano.
    - A referência em REF_TEMP é interpolada por classe de dano.
    - Não usa PCA.
"""

# ============================================================
# 1) IMPORTS
# ============================================================

import os
import re
import time
import copy
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    f1_score
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=UserWarning)

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# 2) CONFIGURAÇÕES
# ============================================================

ARQ_BASE = "base-completo--.pkl"

# Para seus dados, 35 é bom porque D1 e D2 têm 35 °C exato.
# D0 será interpolado entre 32 °C e 38 °C.
REF_TEMP = 35.0

FREQ_MIN_KHZ = 40
FREQ_MAX_KHZ = 50

OUTPUT_DIR = f"TCNN_SUAVE_REF{int(REF_TEMP)}C_{FREQ_MIN_KHZ}-{FREQ_MAX_KHZ}kHz"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Treino
EPOCHS = 450
PATIENCE = 90
BATCH_SIZE = 8
LR = 5e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0

# Correção residual
ALPHA_COMP = 0.85

# Escala máxima do delta no espaço normalizado
RESIDUAL_PERCENTILE = 95.0
RESIDUAL_SCALE_MIN = 0.10
RESIDUAL_SCALE_MAX = 2.50

# Tamanho da suavização do delta
SMOOTH_WIN = 41

# Pesos da loss
LAMBDA_LOW_CURVE = 1.00
LAMBDA_CORR = 0.10
LAMBDA_HF_KEEP = 0.55
LAMBDA_DERIV_KEEP = 0.25
LAMBDA_IDENTITY = 0.35
LAMBDA_DELTA_ENERGY = 0.008
LAMBDA_DELTA_SMOOTH = 0.050
LAMBDA_DAMAGE_IN = 0.05
LAMBDA_DAMAGE_OUT = 0.35
LAMBDA_TEMP = 0.035

# Largura da identidade em °C
IDENTITY_TEMP_WIDTH = 8.0

# Rede
LATENT_DIM = 64
N_COARSE = 128

# Plots
GERAR_PLOTS = True

PLOT_EXEMPLOS = [
    (0, 48.0),
    (1, 55.0),
    (2, 55.0)
]


# ============================================================
# 3) FUNÇÕES BÁSICAS
# ============================================================

def aplicar_estilo_artigo():
    plt.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": 16,
        "axes.labelsize": 18,
        "axes.titlesize": 18,
        "xtick.labelsize": 15,
        "ytick.labelsize": 15,
        "legend.fontsize": 13,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "pdf.fonttype": 42,
        "ps.fonttype": 42
    })


def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None


def get_freq_columns(df, fmin_khz, fmax_khz):
    cols = []
    freqs = []

    for c in df.columns:
        f = extract_freq_hz(c)

        if f is not None:
            f_khz = f / 1e3

            if fmin_khz <= f_khz <= fmax_khz:
                cols.append(c)
                freqs.append(f)

    if len(cols) == 0:
        raise ValueError("Nenhuma coluna de frequência encontrada nessa faixa.")

    order = np.argsort(freqs)

    fcols = [cols[i] for i in order]
    fhz = np.array(freqs, dtype=float)[order]

    return fcols, fhz


def rmsd(y, ref):
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)
    return float(np.sqrt(np.mean((y - ref) ** 2)))


def ccdm(y, ref):
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)

    y0 = y - np.mean(y)
    r0 = ref - np.mean(ref)

    den = np.sqrt(np.sum(y0 ** 2) * np.sum(r0 ** 2)) + 1e-18
    corr = np.sum(y0 * r0) / den

    return float(1.0 - corr)


def moving_average_np(x, win):
    if win <= 1:
        return x.copy()

    if win % 2 == 0:
        win += 1

    win = min(win, len(x) if len(x) % 2 == 1 else len(x) - 1)

    if win < 3:
        return x.copy()

    pad = win // 2
    xp = np.pad(x, (pad, pad), mode="edge")
    kernel = np.ones(win) / win

    return np.convolve(xp, kernel, mode="valid")


# ============================================================
# 4) REFERÊNCIA INTERPOLADA POR DANO
# ============================================================

def curva_mediana_por_temperatura(df_d, fcols):
    temps = np.array(sorted(df_d["temperatura_c"].unique()), dtype=float)

    curvas = []

    for T in temps:
        X_T = df_d.loc[np.isclose(df_d["temperatura_c"], T), fcols].to_numpy(float)
        curvas.append(np.median(X_T, axis=0))

    curvas = np.asarray(curvas, dtype=float)

    return temps, curvas


def interpolar_referencia_em_temp(temps, curvas, ref_temp):
    temps = np.asarray(temps, dtype=float)
    curvas = np.asarray(curvas, dtype=float)

    if np.any(np.isclose(temps, ref_temp)):
        idx = np.where(np.isclose(temps, ref_temp))[0][0]
        return curvas[idx], f"exata {temps[idx]} °C"

    if ref_temp < temps.min() or ref_temp > temps.max():
        idx = np.argmin(np.abs(temps - ref_temp))
        return curvas[idx], f"mais próxima {temps[idx]} °C"

    idx_hi = np.where(temps > ref_temp)[0][0]
    idx_lo = idx_hi - 1

    T_lo = temps[idx_lo]
    T_hi = temps[idx_hi]

    y_lo = curvas[idx_lo]
    y_hi = curvas[idx_hi]

    w = (ref_temp - T_lo) / (T_hi - T_lo)

    y_ref = (1.0 - w) * y_lo + w * y_hi

    return y_ref, f"interpolada entre {T_lo} °C e {T_hi} °C"


def get_reference_curves_by_damage_interpolated(df, fcols, ref_temp):
    ref_by_damage = {}
    info_by_damage = {}

    for d in sorted(df["falha"].unique()):
        df_d = df[df["falha"] == d].copy()

        temps, curvas = curva_mediana_por_temperatura(df_d, fcols)

        y_ref, info = interpolar_referencia_em_temp(
            temps=temps,
            curvas=curvas,
            ref_temp=ref_temp
        )

        ref_by_damage[d] = y_ref
        info_by_damage[d] = info

    return ref_by_damage, info_by_damage


def construir_target_por_dano(df, ref_by_damage):
    Y = []

    for _, row in df.iterrows():
        d = row["falha"]
        Y.append(ref_by_damage[d])

    return np.asarray(Y, dtype=float)


# ============================================================
# 5) SPLIT POR TEMPERATURA
# ============================================================

def make_temperature_group_split(df, ref_temp, val_frac=0.22):
    """
    Divide treino/validação por grupos de temperatura dentro de cada dano.

    Importante:
        O DataFrame precisa estar com index resetado de 0 até N-1.
        Isso já é feito na função executar_tcnn_suave().
    """

    idx_train = []
    idx_val = []

    for d in sorted(df["falha"].unique()):
        df_d = df[df["falha"] == d]
        temps = np.array(sorted(df_d["temperatura_c"].unique()), dtype=float)

        n_val = max(1, int(round(len(temps) * val_frac)))

        candidates = temps[~np.isclose(temps, ref_temp)]

        if len(candidates) == 0:
            candidates = temps

        if len(candidates) <= n_val:
            val_temps = set(candidates.tolist())
        else:
            if len(candidates) >= 4:
                positions = np.linspace(1, len(candidates) - 2, n_val)
            else:
                positions = np.linspace(0, len(candidates) - 1, n_val)

            positions = np.unique(np.round(positions).astype(int))
            val_temps = set(candidates[positions].tolist())

        mask_d = df["falha"] == d
        mask_val = mask_d & df["temperatura_c"].isin(val_temps)

        idx_val.extend(df.index[mask_val].tolist())
        idx_train.extend(df.index[mask_d & (~mask_val)].tolist())

    idx_train = np.array(idx_train, dtype=int)
    idx_val = np.array(idx_val, dtype=int)

    return idx_train, idx_val


# ============================================================
# 6) LOSSES
# ============================================================

def smooth_torch(x, win):
    if win <= 1:
        return x

    if win % 2 == 0:
        win += 1

    L = x.shape[-1]

    if win >= L:
        win = L - 1 if L % 2 == 0 else L

    if win < 3:
        return x

    pad = win // 2
    kernel = torch.ones(1, 1, win, device=x.device, dtype=x.dtype) / win

    xp = F.pad(x, (pad, pad), mode="replicate")
    return F.conv1d(xp, kernel)


def highpass_torch(x, win):
    return x - smooth_torch(x, win)


def corr_loss_batch(y_pred, y_true):
    yp = y_pred.squeeze(1)
    yt = y_true.squeeze(1)

    yp0 = yp - yp.mean(dim=1, keepdim=True)
    yt0 = yt - yt.mean(dim=1, keepdim=True)

    num = torch.sum(yp0 * yt0, dim=1)
    den = torch.sqrt(
        torch.sum(yp0 ** 2, dim=1) *
        torch.sum(yt0 ** 2, dim=1) +
        1e-12
    )

    corr = num / den

    return torch.mean(1.0 - corr)


def derivative_keep_loss(y_pred, x_in):
    dy_pred = y_pred[:, :, 1:] - y_pred[:, :, :-1]
    dy_in = x_in[:, :, 1:] - x_in[:, :, :-1]

    return F.smooth_l1_loss(dy_pred, dy_in)


def delta_smoothness_loss(delta):
    d1 = delta[:, :, 1:] - delta[:, :, :-1]
    return torch.mean(d1 ** 2)


def weighted_smooth_l1(pred, target, weights):
    """
    weights: [B, 1]
    pred/target: [B, 1, L]
    """

    w = weights[:, :, None]
    loss = F.smooth_l1_loss(pred, target, reduction="none")

    return torch.sum(loss * w) / (torch.sum(w) * pred.shape[-1] + 1e-12)


# ============================================================
# 7) MODELO
# ============================================================

class SmoothResidualTCNN(nn.Module):
    """
    Rede pequena para poucos dados.

    Ela não prevê uma curva inteira livremente.
    Ela prevê um delta suave, interpolado a partir de poucos pontos.

    Isso reduz a chance de apagar a assinatura local do dano.
    """

    def __init__(
        self,
        n_points,
        n_classes,
        latent_dim=64,
        n_coarse=128,
        residual_scale=1.0,
        alpha=0.85,
        smooth_win=41
    ):
        super().__init__()

        self.n_points = int(n_points)
        self.n_classes = int(n_classes)
        self.latent_dim = int(latent_dim)
        self.n_coarse = int(min(n_coarse, max(16, n_points // 8)))
        self.residual_scale = float(residual_scale)
        self.alpha = float(alpha)
        self.smooth_win = int(smooth_win)

        def gn(c):
            return nn.GroupNorm(
                num_groups=min(8, c),
                num_channels=c
            )

        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=9, padding=4),
            gn(16),
            nn.GELU(),
            nn.AvgPool1d(2),

            nn.Conv1d(16, 32, kernel_size=9, padding=4),
            gn(32),
            nn.GELU(),
            nn.AvgPool1d(2),

            nn.Conv1d(32, 64, kernel_size=9, padding=4),
            gn(64),
            nn.GELU(),
            nn.AvgPool1d(2),

            nn.Conv1d(64, 96, kernel_size=9, padding=4),
            gn(96),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1)
        )

        self.latent = nn.Sequential(
            nn.Linear(96, 128),
            nn.GELU(),
            nn.Dropout(0.08),
            nn.Linear(128, latent_dim),
            nn.GELU()
        )

        self.delta_head = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.GELU(),
            nn.Dropout(0.08),
            nn.Linear(128, self.n_coarse)
        )

        self.damage_head = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.GELU(),
            nn.Dropout(0.08),
            nn.Linear(64, n_classes)
        )

        self.temp_head = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.GELU(),
            nn.Dropout(0.08),
            nn.Linear(64, 1)
        )

    def encode(self, x):
        h = self.encoder(x).squeeze(-1)
        z = self.latent(h)
        return z

    def make_delta(self, z, L):
        raw_coarse = self.delta_head(z).unsqueeze(1)

        raw_delta = F.interpolate(
            raw_coarse,
            size=L,
            mode="linear",
            align_corners=False
        )

        delta = self.residual_scale * torch.tanh(raw_delta)
        delta = smooth_torch(delta, self.smooth_win)

        return delta

    def forward(self, x):
        L = x.shape[-1]

        z_in = self.encode(x)

        delta = self.make_delta(z_in, L)

        y_comp = x + self.alpha * delta

        z_out = self.encode(y_comp)

        dmg_logits_in = self.damage_head(z_in)
        dmg_logits_out = self.damage_head(z_out)

        temp_pred = self.temp_head(z_in)

        return y_comp, delta, dmg_logits_in, dmg_logits_out, temp_pred


# ============================================================
# 8) PREPARAÇÃO DOS DADOS
# ============================================================

def preparar_dados(df, fcols, ref_by_damage):
    X = df[fcols].to_numpy(float)
    Y = construir_target_por_dano(df, ref_by_damage)

    falhas_orig = df["falha"].to_numpy()
    classes = sorted(np.unique(falhas_orig))

    class_to_idx = {c: i for i, c in enumerate(classes)}
    idx_to_class = {i: c for c, i in class_to_idx.items()}

    y_class = np.array(
        [class_to_idx[c] for c in falhas_orig],
        dtype=int
    )

    T = df["temperatura_c"].to_numpy(float)
    T_aux = ((T - REF_TEMP) / 100.0).reshape(-1, 1)

    identity_w = np.exp(
        -np.abs(T - REF_TEMP) / IDENTITY_TEMP_WIDTH
    ).reshape(-1, 1)

    return X, Y, y_class, T_aux, identity_w, class_to_idx, idx_to_class


# ============================================================
# 9) TREINO
# ============================================================

def train_tcnn_suave(df, fcols, ref_by_damage):
    print("\n====================================================")
    print("TREINANDO TCNN SUAVE")
    print("====================================================")
    print("Entrada da rede: somente curva")
    print("Falha e temperatura: apenas alvos auxiliares")
    print("Correção: residual suave")
    print("Alvo principal: referência interpolada da mesma classe de dano")

    X, Y, y_class, T_aux, identity_w, class_to_idx, idx_to_class = preparar_dados(
        df=df,
        fcols=fcols,
        ref_by_damage=ref_by_damage
    )

    idx_train, idx_val = make_temperature_group_split(
        df=df,
        ref_temp=REF_TEMP,
        val_frac=0.22
    )

    print(f"\nAmostras treino: {len(idx_train)}")
    print(f"Amostras validação: {len(idx_val)}")

    scaler = StandardScaler()
    scaler.fit(
        np.vstack([
            X[idx_train],
            Y[idx_train]
        ])
    )

    Xs = scaler.transform(X)
    Ys = scaler.transform(Y)

    residual_train = Ys[idx_train] - Xs[idx_train]

    residual_scale = float(
        np.percentile(
            np.abs(residual_train),
            RESIDUAL_PERCENTILE
        )
    )

    residual_scale = float(
        np.clip(
            residual_scale,
            RESIDUAL_SCALE_MIN,
            RESIDUAL_SCALE_MAX
        )
    )

    print(f"Residual scale usado: {residual_scale:.4f}")

    X_tensor = torch.tensor(Xs[:, None, :], dtype=torch.float32)
    Y_tensor = torch.tensor(Ys[:, None, :], dtype=torch.float32)
    C_tensor = torch.tensor(y_class, dtype=torch.long)
    T_tensor = torch.tensor(T_aux, dtype=torch.float32)
    W_tensor = torch.tensor(identity_w, dtype=torch.float32)

    train_ds = TensorDataset(
        X_tensor[idx_train],
        Y_tensor[idx_train],
        C_tensor[idx_train],
        T_tensor[idx_train],
        W_tensor[idx_train]
    )

    val_ds = TensorDataset(
        X_tensor[idx_val],
        Y_tensor[idx_val],
        C_tensor[idx_val],
        T_tensor[idx_val],
        W_tensor[idx_val]
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Dispositivo: {device}")

    model = SmoothResidualTCNN(
        n_points=X.shape[1],
        n_classes=len(class_to_idx),
        latent_dim=LATENT_DIM,
        n_coarse=N_COARSE,
        residual_scale=residual_scale,
        alpha=ALPHA_COMP,
        smooth_win=SMOOTH_WIN
    ).to(device)

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="min",
        factor=0.5,
        patience=35
    )

    ce_loss = nn.CrossEntropyLoss()
    mse_loss = nn.MSELoss()
    huber = nn.SmoothL1Loss()

    best_val = np.inf
    best_state = copy.deepcopy(model.state_dict())
    no_improve = 0

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "val_low_curve": [],
        "val_hf_keep": [],
        "val_damage_acc_out": [],
        "lr": []
    }

    t0 = time.time()

    for ep in range(1, EPOCHS + 1):
        model.train()
        train_losses = []

        for xb, yb, cb, tb, wb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            cb = cb.to(device)
            tb = tb.to(device)
            wb = wb.to(device)

            opt.zero_grad()

            pred, delta, logits_in, logits_out, tpred = model(xb)

            pred_low = smooth_torch(pred, SMOOTH_WIN)
            yb_low = smooth_torch(yb, SMOOTH_WIN)

            pred_hf = highpass_torch(pred, SMOOTH_WIN)
            xb_hf = highpass_torch(xb, SMOOTH_WIN)

            loss_low_curve = huber(pred_low, yb_low)
            loss_corr = corr_loss_batch(pred, yb)

            loss_hf_keep = huber(pred_hf, xb_hf)
            loss_deriv_keep = derivative_keep_loss(pred, xb)

            loss_identity = weighted_smooth_l1(pred, xb, wb)

            loss_delta_energy = torch.mean(delta ** 2)
            loss_delta_smooth = delta_smoothness_loss(delta)

            loss_dmg_in = ce_loss(logits_in, cb)
            loss_dmg_out = ce_loss(logits_out, cb)

            loss_temp = mse_loss(tpred, tb)

            loss = (
                LAMBDA_LOW_CURVE * loss_low_curve
                + LAMBDA_CORR * loss_corr
                + LAMBDA_HF_KEEP * loss_hf_keep
                + LAMBDA_DERIV_KEEP * loss_deriv_keep
                + LAMBDA_IDENTITY * loss_identity
                + LAMBDA_DELTA_ENERGY * loss_delta_energy
                + LAMBDA_DELTA_SMOOTH * loss_delta_smooth
                + LAMBDA_DAMAGE_IN * loss_dmg_in
                + LAMBDA_DAMAGE_OUT * loss_dmg_out
                + LAMBDA_TEMP * loss_temp
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0
            )

            opt.step()

            train_losses.append(float(loss.item()))

        train_loss = float(np.mean(train_losses))

        model.eval()

        val_losses = []
        val_low_losses = []
        val_hf_losses = []

        pred_classes_out = []
        true_classes = []

        with torch.no_grad():
            for xb, yb, cb, tb, wb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                cb = cb.to(device)
                tb = tb.to(device)
                wb = wb.to(device)

                pred, delta, logits_in, logits_out, tpred = model(xb)

                pred_low = smooth_torch(pred, SMOOTH_WIN)
                yb_low = smooth_torch(yb, SMOOTH_WIN)

                pred_hf = highpass_torch(pred, SMOOTH_WIN)
                xb_hf = highpass_torch(xb, SMOOTH_WIN)

                loss_low_curve = huber(pred_low, yb_low)
                loss_corr = corr_loss_batch(pred, yb)

                loss_hf_keep = huber(pred_hf, xb_hf)
                loss_deriv_keep = derivative_keep_loss(pred, xb)

                loss_identity = weighted_smooth_l1(pred, xb, wb)

                loss_delta_energy = torch.mean(delta ** 2)
                loss_delta_smooth = delta_smoothness_loss(delta)

                loss_dmg_in = ce_loss(logits_in, cb)
                loss_dmg_out = ce_loss(logits_out, cb)

                loss_temp = mse_loss(tpred, tb)

                loss = (
                    LAMBDA_LOW_CURVE * loss_low_curve
                    + LAMBDA_CORR * loss_corr
                    + LAMBDA_HF_KEEP * loss_hf_keep
                    + LAMBDA_DERIV_KEEP * loss_deriv_keep
                    + LAMBDA_IDENTITY * loss_identity
                    + LAMBDA_DELTA_ENERGY * loss_delta_energy
                    + LAMBDA_DELTA_SMOOTH * loss_delta_smooth
                    + LAMBDA_DAMAGE_IN * loss_dmg_in
                    + LAMBDA_DAMAGE_OUT * loss_dmg_out
                    + LAMBDA_TEMP * loss_temp
                )

                val_losses.append(float(loss.item()))
                val_low_losses.append(float(loss_low_curve.item()))
                val_hf_losses.append(float(loss_hf_keep.item()))

                pred_c = torch.argmax(logits_out, dim=1)
                pred_classes_out.extend(pred_c.cpu().numpy().tolist())
                true_classes.extend(cb.cpu().numpy().tolist())

        val_loss = float(np.mean(val_losses))
        val_low = float(np.mean(val_low_losses))
        val_hf = float(np.mean(val_hf_losses))
        val_acc_out = accuracy_score(true_classes, pred_classes_out)

        scheduler.step(val_loss)

        current_lr = opt.param_groups[0]["lr"]

        history["epoch"].append(ep)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_low_curve"].append(val_low)
        history["val_hf_keep"].append(val_hf)
        history["val_damage_acc_out"].append(val_acc_out)
        history["lr"].append(current_lr)

        if val_loss < best_val - 1e-7:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1

        if ep == 1 or ep % 25 == 0:
            elapsed = time.time() - t0
            print(
                f"Epoch {ep:4d}/{EPOCHS} | "
                f"train={train_loss:.6f} | "
                f"val={val_loss:.6f} | "
                f"low={val_low:.6f} | "
                f"hf={val_hf:.6f} | "
                f"dmg_out_acc={val_acc_out:.3f} | "
                f"lr={current_lr:.2e} | "
                f"tempo={elapsed:.1f}s"
            )

        if no_improve >= PATIENCE:
            print(
                f"Early stopping na epoch {ep}. "
                f"Melhor val_loss = {best_val:.6f}"
            )
            break

    model.load_state_dict(best_state)

    history = pd.DataFrame(history)

    extra = {
        "model": model,
        "scaler": scaler,
        "device": device,
        "history": history,
        "class_to_idx": class_to_idx,
        "idx_to_class": idx_to_class,
        "idx_train": idx_train,
        "idx_val": idx_val,
        "residual_scale": residual_scale
    }

    return extra


# ============================================================
# 10) APLICAR MODELO
# ============================================================

def aplicar_tcnn_suave(df, fcols, extra):
    model = extra["model"]
    scaler = extra["scaler"]
    device = extra["device"]
    idx_to_class = extra["idx_to_class"]

    X = df[fcols].to_numpy(float)
    Xs = scaler.transform(X)

    X_tensor = torch.tensor(
        Xs[:, None, :],
        dtype=torch.float32
    )

    preds_scaled = []
    deltas_scaled = []
    logits_out_all = []
    temp_pred_all = []

    model.eval()

    with torch.no_grad():
        for i in range(0, len(X_tensor), BATCH_SIZE):
            xb = X_tensor[i:i + BATCH_SIZE].to(device)

            pred, delta, logits_in, logits_out, tpred = model(xb)

            preds_scaled.append(pred.cpu().numpy()[:, 0, :])
            deltas_scaled.append(delta.cpu().numpy()[:, 0, :])
            logits_out_all.append(logits_out.cpu().numpy())
            temp_pred_all.append(tpred.cpu().numpy())

    preds_scaled = np.vstack(preds_scaled)
    deltas_scaled = np.vstack(deltas_scaled)
    logits_out_all = np.vstack(logits_out_all)
    temp_pred_all = np.vstack(temp_pred_all)

    Y_comp = scaler.inverse_transform(preds_scaled)

    pred_class_idx = np.argmax(logits_out_all, axis=1)
    pred_damage = np.array(
        [idx_to_class[i] for i in pred_class_idx]
    )

    pred_temp_c = temp_pred_all[:, 0] * 100.0 + REF_TEMP

    df_comp = df.copy()
    df_comp[fcols] = Y_comp
    df_comp["falha_pred_aux"] = pred_damage
    df_comp["temperatura_pred_aux"] = pred_temp_c

    return df_comp


# ============================================================
# 11) MÉTRICAS E RESUMOS
# ============================================================

def calcular_metricas(df_curvas, fcols, y_ref_healthy, ref_by_damage, metodo):
    X = df_curvas[fcols].to_numpy(float)

    out = df_curvas.copy()

    out["RMSD"] = [rmsd(x, y_ref_healthy) for x in X]
    out["CCDM"] = [ccdm(x, y_ref_healthy) for x in X]

    rmsd_class = []
    ccdm_class = []

    for i, (_, row) in enumerate(df_curvas.iterrows()):
        d = row["falha"]
        ref_d = ref_by_damage[d]

        rmsd_class.append(rmsd(X[i], ref_d))
        ccdm_class.append(ccdm(X[i], ref_d))

    out["RMSD_class_ref"] = rmsd_class
    out["CCDM_class_ref"] = ccdm_class
    out["Metodo"] = metodo

    return out


def resumo_geral(df_long):
    tabela = (
        df_long
        .groupby(["Metodo", "falha"])[[
            "RMSD",
            "CCDM",
            "RMSD_class_ref",
            "CCDM_class_ref"
        ]]
        .agg(["mean", "std", "min", "max"])
        .round(6)
    )

    return tabela


def checar_ordem_media_por_dano(df_long, metodo="TCNN_Suave"):
    df_m = df_long[df_long["Metodo"] == metodo].copy()

    g = (
        df_m
        .groupby("falha")[["RMSD", "CCDM"]]
        .mean()
        .reset_index()
        .sort_values("falha")
    )

    print("\n================ ORDEM MÉDIA POR DANO ================")
    print(g)

    if set(g["falha"]) >= {0, 1, 2}:
        vals_r = dict(zip(g["falha"], g["RMSD"]))
        vals_c = dict(zip(g["falha"], g["CCDM"]))

        print("\nRMSD D0 < D1 < D2 ?", vals_r[0] < vals_r[1] < vals_r[2])
        print("CCDM D0 < D1 < D2 ?", vals_c[0] < vals_c[1] < vals_c[2])

    return g


def avaliar_auxiliares(df_comp):
    y_true = df_comp["falha"].to_numpy()
    y_pred = df_comp["falha_pred_aux"].to_numpy()

    print("\n================ CLASSIFICAÇÃO AUXILIAR DO DANO ================")
    print("Essa classificação é auxiliar. A falha NÃO entrou como input.")
    print(confusion_matrix(y_true, y_pred))
    print(f"ACC = {accuracy_score(y_true, y_pred):.4f}")
    print(f"F1 macro = {f1_score(y_true, y_pred, average='macro'):.4f}")
    print(classification_report(y_true, y_pred, digits=4))

    temp_true = df_comp["temperatura_c"].to_numpy(float)
    temp_pred = df_comp["temperatura_pred_aux"].to_numpy(float)

    mae = np.mean(np.abs(temp_true - temp_pred))
    rmse = np.sqrt(np.mean((temp_true - temp_pred) ** 2))

    print("\n================ PREDIÇÃO AUXILIAR DE TEMPERATURA ================")
    print("A temperatura NÃO entrou como input.")
    print(f"MAE temperatura = {mae:.4f} °C")
    print(f"RMSE temperatura = {rmse:.4f} °C")


# ============================================================
# 12) PLOTS
# ============================================================

def selecionar_indice_exemplo(df, falha, temperatura):
    df_d = df[df["falha"] == falha].copy()

    if len(df_d) == 0:
        raise ValueError(f"Não existe falha={falha} no dataset.")

    df_t = df_d[np.isclose(df_d["temperatura_c"], temperatura)]

    if len(df_t) > 0:
        return df_t.index[0]

    return (df_d["temperatura_c"] - temperatura).abs().idxmin()


def plot_loss(history, salvar=True):
    aplicar_estilo_artigo()

    plt.figure(figsize=(10, 5))

    plt.plot(
        history["epoch"],
        history["train_loss"],
        lw=2,
        label="Treino"
    )

    plt.plot(
        history["epoch"],
        history["val_loss"],
        lw=2,
        label="Validação total"
    )

    plt.plot(
        history["epoch"],
        history["val_low_curve"],
        lw=2,
        label="Validação baixa frequência"
    )

    plt.plot(
        history["epoch"],
        history["val_hf_keep"],
        lw=2,
        label="Preservação alta frequência"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Treinamento da TCNN Suave")

    plt.grid(False)

    ax = plt.gca()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.legend(
        frameon=True,
        facecolor="white",
        edgecolor="none"
    )

    plt.tight_layout()

    if salvar:
        path = os.path.join(OUTPUT_DIR, "loss_tcnn_suave.png")
        plt.savefig(path, bbox_inches="tight", facecolor="white")
        print(f"Figura salva em: {path}")

    plt.show()


def plot_metricas_por_dano(df_long, salvar=True):
    aplicar_estilo_artigo()

    metricas = [
        "RMSD",
        "CCDM",
        "RMSD_class_ref",
        "CCDM_class_ref"
    ]

    for metrica in metricas:
        plt.figure(figsize=(9, 5))

        for metodo in sorted(df_long["Metodo"].unique()):
            df_m = df_long[df_long["Metodo"] == metodo]

            g = (
                df_m
                .groupby("falha")[metrica]
                .mean()
                .reset_index()
                .sort_values("falha")
            )

            plt.plot(
                g["falha"],
                g[metrica],
                marker="o",
                lw=2,
                label=metodo
            )

        plt.xlabel("Classe de dano")
        plt.ylabel(metrica)
        plt.title(f"{metrica} médio por dano")
        plt.xticks(sorted(df_long["falha"].unique()))

        plt.grid(False)

        ax = plt.gca()
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        plt.legend(
            frameon=True,
            facecolor="white",
            edgecolor="none"
        )

        plt.tight_layout()

        if salvar:
            path = os.path.join(OUTPUT_DIR, f"{metrica}_medio_por_dano.png")
            plt.savefig(path, bbox_inches="tight", facecolor="white")
            print(f"Figura salva em: {path}")

        plt.show()


def plot_metricas_por_temperatura(df_long, salvar=True):
    aplicar_estilo_artigo()

    for dano in sorted(df_long["falha"].unique()):
        df_d = df_long[df_long["falha"] == dano].copy()

        fig, axes = plt.subplots(
            1,
            2,
            figsize=(15, 5.5),
            dpi=300
        )

        for ax, metrica in zip(axes, ["RMSD", "CCDM"]):
            for metodo in sorted(df_d["Metodo"].unique()):
                df_m = df_d[df_d["Metodo"] == metodo]

                g = (
                    df_m
                    .groupby("temperatura_c")[metrica]
                    .mean()
                    .reset_index()
                    .sort_values("temperatura_c")
                )

                ax.plot(
                    g["temperatura_c"],
                    g[metrica],
                    marker="o",
                    lw=2,
                    label=metodo
                )

            ax.set_xlabel("Temperatura (°C)")
            ax.set_ylabel(metrica)
            ax.set_title(f"{metrica} — Dano {dano}")

            ax.grid(False)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

        axes[0].legend(
            frameon=True,
            facecolor="white",
            edgecolor="none"
        )

        plt.tight_layout()

        if salvar:
            path = os.path.join(
                OUTPUT_DIR,
                f"metricas_por_temperatura_dano_{dano}.png"
            )
            plt.savefig(path, bbox_inches="tight", facecolor="white")
            print(f"Figura salva em: {path}")

        plt.show()


def plot_curvas_exemplo(
    df_base,
    df_comp,
    ref_by_damage,
    fcols,
    fhz,
    exemplos,
    salvar=True
):
    aplicar_estilo_artigo()

    fhz_khz = fhz / 1e3

    for falha, temperatura in exemplos:
        idx = selecionar_indice_exemplo(
            df=df_base,
            falha=falha,
            temperatura=temperatura
        )

        T_real = df_base.loc[idx, "temperatura_c"]
        D_real = df_base.loc[idx, "falha"]

        y_orig = df_base.loc[idx, fcols].to_numpy(float)
        y_comp = df_comp.loc[idx, fcols].to_numpy(float)

        y_ref_class = ref_by_damage[D_real]
        y_ref_healthy = ref_by_damage[0]

        print("\n================ CURVA EXEMPLO ================")
        print(f"Índice interno: {idx}")

        if "indice_original" in df_base.columns:
            print(f"Índice original: {df_base.loc[idx, 'indice_original']}")

        print(f"Falha: {D_real}")
        print(f"Temperatura: {T_real} °C")

        print(
            "Original vs ref classe: "
            f"RMSD={rmsd(y_orig, y_ref_class):.6f}, "
            f"CCDM={ccdm(y_orig, y_ref_class):.6f}"
        )

        print(
            "Compensada vs ref classe: "
            f"RMSD={rmsd(y_comp, y_ref_class):.6f}, "
            f"CCDM={ccdm(y_comp, y_ref_class):.6f}"
        )

        print(
            "Compensada vs ref saudável: "
            f"RMSD={rmsd(y_comp, y_ref_healthy):.6f}, "
            f"CCDM={ccdm(y_comp, y_ref_healthy):.6f}"
        )

        plt.figure(figsize=(12, 5.8), dpi=300)

        plt.plot(
            fhz_khz,
            y_ref_healthy,
            "--",
            c="black",
            lw=1.2,
            label=f"Referência saudável {REF_TEMP:.0f} °C"
        )

        if D_real != 0:
            plt.plot(
                fhz_khz,
                y_ref_class,
                "-.",
                c="gray",
                lw=1.2,
                label=f"Referência D{D_real} {REF_TEMP:.0f} °C"
            )

        plt.plot(
            fhz_khz,
            y_orig,
            lw=1.0,
            alpha=0.65,
            label=f"Original — D{D_real} — {T_real:.0f} °C"
        )

        plt.plot(
            fhz_khz,
            y_comp,
            lw=2.0,
            label="TCNN Suave compensada"
        )

        plt.xlabel("Frequência (kHz)")
        plt.ylabel("Parte real da impedância")
        plt.title(f"Compensação térmica — Dano {D_real} — {T_real:.0f} °C")

        plt.grid(False)

        ax = plt.gca()
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        plt.legend(
            frameon=True,
            facecolor="white",
            edgecolor="none"
        )

        plt.tight_layout()

        if salvar:
            path = os.path.join(
                OUTPUT_DIR,
                f"curva_exemplo_D{D_real}_T{T_real:.0f}_idx{idx}.png"
            )

            plt.savefig(
                path,
                bbox_inches="tight",
                facecolor="white"
            )

            print(f"Figura salva em: {path}")

        plt.show()


# ============================================================
# 13) EXECUÇÃO PRINCIPAL
# ============================================================

def executar_tcnn_suave():
    t_total = time.time()

    print("====================================================")
    print("TCNN SUAVE — COMPENSAÇÃO TÉRMICA RESIDUAL")
    print("====================================================")

    df_raw = pd.read_pickle(ARQ_BASE)

    df = df_raw.copy()
    df["indice_original"] = df.index
    df = df.reset_index(drop=True)

    required = {"temperatura_c", "falha"}
    missing = required - set(df.columns)

    if len(missing) > 0:
        raise ValueError(f"Colunas obrigatórias ausentes: {missing}")

    fcols, fhz = get_freq_columns(
        df=df,
        fmin_khz=FREQ_MIN_KHZ,
        fmax_khz=FREQ_MAX_KHZ
    )

    print(f"\nTotal de amostras: {len(df)}")
    print(f"Classes de dano: {sorted(df['falha'].unique())}")
    print("Temperaturas por dano:")

    for d in sorted(df["falha"].unique()):
        temps = sorted(df.loc[df["falha"] == d, "temperatura_c"].unique())
        print(f"  Dano {d}: {temps}")

    print(f"\nFaixa usada: {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
    print(f"Número de pontos de frequência: {len(fcols)}")
    print(f"Temperatura de referência: {REF_TEMP} °C")

    ref_by_damage, info_by_damage = get_reference_curves_by_damage_interpolated(
        df=df,
        fcols=fcols,
        ref_temp=REF_TEMP
    )

    print("\nReferência usada por classe:")

    for d, info in info_by_damage.items():
        print(f"  Dano {d}: {info}")

    y_ref_healthy = ref_by_damage[0]

    df_original_metricas = calcular_metricas(
        df_curvas=df,
        fcols=fcols,
        y_ref_healthy=y_ref_healthy,
        ref_by_damage=ref_by_damage,
        metodo="Original"
    )

    extra = train_tcnn_suave(
        df=df,
        fcols=fcols,
        ref_by_damage=ref_by_damage
    )

    df_comp = aplicar_tcnn_suave(
        df=df,
        fcols=fcols,
        extra=extra
    )

    df_comp_metricas = calcular_metricas(
        df_curvas=df_comp,
        fcols=fcols,
        y_ref_healthy=y_ref_healthy,
        ref_by_damage=ref_by_damage,
        metodo="TCNN_Suave"
    )

    df_long = pd.concat(
        [
            df_original_metricas,
            df_comp_metricas
        ],
        axis=0,
        ignore_index=False
    )

    tabela_resumo = resumo_geral(df_long)

    ordem_media = checar_ordem_media_por_dano(
        df_long,
        metodo="TCNN_Suave"
    )

    print("\n================ RESUMO GERAL ================")
    print(tabela_resumo)

    avaliar_auxiliares(df_comp)

    # Salvar resultados
    df_comp.to_pickle(
        os.path.join(
            OUTPUT_DIR,
            "df_tcnn_suave_compensado.pkl"
        )
    )

    df_comp.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "df_tcnn_suave_compensado.csv"
        ),
        index=False
    )

    df_long.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "df_metricas_original_tcnn_suave.csv"
        ),
        index=False
    )

    extra["history"].to_csv(
        os.path.join(
            OUTPUT_DIR,
            "historico_tcnn_suave.csv"
        ),
        index=False
    )

    tabela_resumo.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "tabela_resumo.csv"
        )
    )

    ordem_media.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "ordem_media_por_dano.csv"
        ),
        index=False
    )

    # Plots
    if GERAR_PLOTS:
        plot_loss(
            history=extra["history"],
            salvar=True
        )

        plot_metricas_por_dano(
            df_long=df_long,
            salvar=True
        )

        plot_metricas_por_temperatura(
            df_long=df_long,
            salvar=True
        )

        plot_curvas_exemplo(
            df_base=df,
            df_comp=df_comp,
            ref_by_damage=ref_by_damage,
            fcols=fcols,
            fhz=fhz,
            exemplos=PLOT_EXEMPLOS,
            salvar=True
        )

    tempo_total = time.time() - t_total

    print("\n====================================================")
    print("FINALIZADO")
    print("====================================================")
    print(f"Tempo total: {tempo_total:.2f} s")
    print(f"Resultados salvos em: {OUTPUT_DIR}")

    return {
        "df_base": df,
        "df_comp": df_comp,
        "df_long": df_long,
        "tabela_resumo": tabela_resumo,
        "ordem_media": ordem_media,
        "ref_by_damage": ref_by_damage,
        "info_by_damage": info_by_damage,
        "fcols": fcols,
        "fhz": fhz,
        "extra": extra,
        "history": extra["history"]
    }


# ============================================================
# 14) RODAR
# ============================================================

resultados = executar_tcnn_suave()

df_base = resultados["df_base"]
df_comp = resultados["df_comp"]
df_long = resultados["df_long"]
tabela_resumo = resultados["tabela_resumo"]
ordem_media = resultados["ordem_media"]
ref_by_damage = resultados["ref_by_damage"]
info_by_damage = resultados["info_by_damage"]
fcols = resultados["fcols"]
fhz = resultados["fhz"]
extra = resultados["extra"]
history = resultados["history"]

print("\n================ TABELA RESUMO ================")
display(tabela_resumo)

print("\n================ ORDEM MÉDIA POR DANO ================")
display(ordem_media)